# Experiment 07: All-26-Layers Activation Frequency-Guided Mutually Exclusive Tensor Compression (Branch 1)

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25].mlp.gate_proj`)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Key Pipeline Principles:
1. **Model-Wide Activation Profiling**: Simultaneous forward hooks across all 26 decoder layers during uncompressed baseline inference ($W_{\text{orig}}$).
2. **Layer-Wise Activation Outlier Filter**: Quarantines extreme superweights ($|x| > 3.0$ or top 1% variance) independently for each layer in uncompressed FP32.
3. **1D Activation Frequency Metric (Branch 1)**: For each layer and normal active coordinate, extracts:
   $$\{c: \text{Top\_most\_K\_frequent\_activation\_value}\} \quad (v_c \in \mathbb{R})$$
4. **1D Absolute Difference Clustering**: Partitions coordinates into mutually exclusive clusters ($C_i \cap C_j = \emptyset$) minimizing intra-cluster distance $d(x, y) = |v_x - v_y|$.
5. **Sequential Layer-by-Layer Tucker Factorization**: Stacks cluster slices into 3D tensors $\mathcal{T}_l \in \mathbb{R}^{6 \times 400 \times 1152}$ and decomposes via Tucker.
6. **Full-Model Evaluation**: Evaluates overall task accuracy and aggregate parameter reduction with all 26 layers compressed simultaneously.

In [1]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    from neural_decomp.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from neural_decomp.utils import get_device_info, save_json_metrics
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions
    from utility.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from utility.utils import get_device_info, save_json_metrics
    print("Loaded utility library.")

device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")

/home/dwithun/Development/llm_compression/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded neural_decomp library.
Device: NVIDIA GeForce RTX 3070 Ti | CUDA Available: True


In [2]:
# =====================================================================
# STEP 2: Initialize Model & Tokenizer Across All 26 Layers
# =====================================================================
model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

NUM_LAYERS = len(model.model.layers)
print(f"Loaded model with {NUM_LAYERS} transformer decoder layers.")

# Save pristine unprocessed weights for all 26 gate_proj layers for rollbacks
W_gate_orig_all = {
    l: model.model.layers[l].mlp.gate_proj.weight.data.clone()
    for l in range(NUM_LAYERS)
}
print(f"Stored pristine baseline weights for all {NUM_LAYERS} gate_proj modules.")

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 377.79it/s]


Loaded model with 26 transformer decoder layers.
Stored pristine baseline weights for all 26 gate_proj modules.
time: 4.24s
cummulative_time: 6.09s


In [3]:
# =====================================================================
# STEP 3: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

# Calibration & benchmark evaluation subset
EVAL_SAMPLE_COUNT = 150
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI: {len(ds):,} total samples | Evaluation Subset: {len(eval_data):,} samples")

Loaded GLUE MNLI: 9,815 total samples | Evaluation Subset: 150 samples
time: 3.32s
cummulative_time: 9.41s


## Step 1: Model-Wide Activation Profiling Across All 26 Layers

We attach forward hooks to `model.model.layers[l].mlp.act_fn` for all $l \in [0, 25]$ simultaneously during the initial uncompressed benchmark pass.
Each hook mean-pools sequence activations to capture a $[6912]$ empirical trajectory per sample per layer.

In [4]:
# =====================================================================
# STEP 4: Initial Baseline Evaluation & Simultaneous 26-Layer Hooking
# =====================================================================
layer_trajectories = {l: [] for l in range(NUM_LAYERS)}
current_acts = {}

def make_act_hook(layer_idx):
    def hook_fn(module, input_tensor, output_tensor):
        act = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
        current_acts[layer_idx] = act.detach().cpu()
    return hook_fn

hooks = [
    model.model.layers[l].mlp.act_fn.register_forward_hook(make_act_hook(l))
    for l in range(NUM_LAYERS)
]

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Baseline Eval & 26-Layer Activation Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        # Store mean-pooled activation vector for all 26 layers
        for l in range(NUM_LAYERS):
            if l in current_acts and current_acts[l] is not None:
                pooled = current_acts[l].squeeze(0).mean(dim=0).numpy()
                layer_trajectories[l].append(pooled)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

# Remove all hooks
for h in hooks:
    h.remove()

# Convert to numpy matrices: {layer: [num_samples, 6912]}
acts_matrices = {l: np.stack(layer_trajectories[l]) for l in range(NUM_LAYERS)}

baseline_accuracy = accuracy_score(ground_truth, predictions)
print(f"\nUncompressed Baseline Accuracy (All Layers): {baseline_accuracy * 100:.2f}%")
print(f"Profiled activation matrices for {len(acts_matrices)} layers (shape per layer: {acts_matrices[0].shape}).")

Baseline Eval & 26-Layer Activation Profiling: 100%|██████████| 150/150 [00:07<00:00, 19.35it/s]



Uncompressed Baseline Accuracy (All Layers): 50.67%
Profiled activation matrices for 26 layers (shape per layer: (150, 6912)).
time: 7.83s
cummulative_time: 17.24s


## Step 2: Layer-by-Layer Branch 1 Decomposition Loop ($l = 0 \dots 25$)

For each layer $l \in [0, 25]$:
1. **Isolate Superweights**: Identify outlier coordinates ($|x| > 3.0$ or top 1% variance) to retain in pristine FP32.
2. **Extract 1D Top Frequent Values**: Find modal activation value $v_c \in \mathbb{R}$ per normal active coordinate.
3. **Mutually Exclusive Grouping**: Sort along 1D scalar and partition into $N=6$ clusters of 400 coordinates ($2,400$ total active coordinates).
4. **3D Tensor Construction & Tucker**: Stack into $\mathcal{T}_l \in \mathbb{R}^{6 \times 400 \times 1152}$ and decompose with Moderate ranks $[4, 180, 600]$.
5. **Reconstruct & Inject**: Inject compressed weight slices back into `model.model.layers[l].mlp.gate_proj` while preserving superweights.

In [5]:
# =====================================================================
# STEP 5: Execute Branch 1 Compression Sequentially Across All 26 Layers
# =====================================================================
NUM_CLUSTERS = 6
COORDS_PER_CLUSTER = 400
TARGET_TOTAL_ACTIVE = NUM_CLUSTERS * COORDS_PER_CLUSTER
TUCKER_RANKS = [4, 180, 600]

layer_compression_records = []
total_orig_params_gate = 0
total_comp_params_gate = 0

print(f"{'Layer':<6} | {'Superweights':<13} | {'Recon Err %':<12} | {'Params Cut %':<12} | {'Ratio':<6}")
print("-" * 65)

for l in range(NUM_LAYERS):
    acts_l = acts_matrices[l]
    W_gate_l = W_gate_orig_all[l]
    num_coords = W_gate_l.shape[0]
    d_in = W_gate_l.shape[1]
    
    # 1. Outlier Filter per layer
    max_mags = np.max(np.abs(acts_l), axis=0)
    variances = np.var(acts_l, axis=0)
    super_mask = (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]
    
    inactive_mask = (np.mean(np.abs(acts_l) < 0.05, axis=0) > 0.90) & (~super_mask)
    normal_active_indices = np.where((~super_mask) & (~inactive_mask))[0]
    
    # 2. Extract 1D Top Frequent Values (Branch 1)
    rounded_acts = np.round(acts_l[:, normal_active_indices], decimals=1)
    top_vals = []
    for idx in range(len(normal_active_indices)):
        vals, counts = np.unique(rounded_acts[:, idx], return_counts=True)
        top_vals.append(vals[np.argmax(counts)])
    top_vals = np.array(top_vals)
    
    # 3. Mutually Exclusive 1D Absolute Difference Grouping
    sorted_order = np.argsort(top_vals)[:TARGET_TOTAL_ACTIVE]
    selected_coords = normal_active_indices[sorted_order]
    
    cluster_slices = []
    clusters_dict = {}
    for k in range(NUM_CLUSTERS):
        c_coords = selected_coords[k * COORDS_PER_CLUSTER : (k + 1) * COORDS_PER_CLUSTER]
        clusters_dict[k] = c_coords
        cluster_slices.append(W_gate_l[c_coords, :].float().cpu())
        
    # 4. Assemble 3D Tensor T_l & Tucker Decomposition
    T_l = torch.stack(cluster_slices, dim=0)
    core, factors = tucker(T_l, rank=TUCKER_RANKS, init='svd')
    T_l_recon = tucker_to_tensor((core, factors))
    
    # Reconstruction error on active tensor
    W_orig_stacked = T_l.reshape(-1, d_in)
    W_recon_stacked = T_l_recon.reshape(-1, d_in)
    rel_err = (torch.norm(W_orig_stacked - W_recon_stacked) / torch.norm(W_orig_stacked)).item()
    
    # Parameter Accounting for this layer's gate_proj
    core_params = core.numel()
    factor_params = sum(f.numel() for f in factors)
    comp_active_params = core_params + factor_params
    orig_active_params = T_l.numel()
    
    # Full layer gate_proj accounting
    full_orig_params = W_gate_l.numel()
    full_comp_params = full_orig_params - orig_active_params + comp_active_params
    layer_cut_pct = (1.0 - full_comp_params / full_orig_params) * 100.0
    layer_ratio = full_orig_params / full_comp_params
    
    total_orig_params_gate += full_orig_params
    total_comp_params_gate += full_comp_params
    
    # 5. Inject Reconstructed Weights into Model Layer l
    W_reconstructed = W_gate_l.clone().float().cpu()
    for k in range(NUM_CLUSTERS):
        W_reconstructed[clusters_dict[k], :] = T_l_recon[k].float().cpu()
    W_reconstructed[super_indices, :] = W_gate_l[super_indices, :].float().cpu()
    
    target_mod = model.model.layers[l].mlp.gate_proj
    target_mod.weight.data = W_reconstructed.to(device=model.device, dtype=target_mod.weight.dtype)
    
    layer_record = {
        "layer": l,
        "superweights": len(super_indices),
        "recon_error": rel_err,
        "layer_cut_pct": layer_cut_pct,
        "layer_ratio": layer_ratio,
    }
    layer_compression_records.append(layer_record)
    
    if l % 5 == 0 or l == NUM_LAYERS - 1:
        print(f"L{l:<4} | {len(super_indices):<13} | {rel_err * 100:>10.2f}% | {layer_cut_pct:>10.2f}% | {layer_ratio:>5.2f}x")

print("-" * 65)
print(f"All {NUM_LAYERS} layers compressed and injected successfully.")

Layer  | Superweights  | Recon Err %  | Params Cut % | Ratio 
-----------------------------------------------------------------
L0    | 71            |      77.11% |      19.71% |  1.25x
L5    | 70            |      76.66% |      19.71% |  1.25x
L10   | 70            |      74.87% |      19.71% |  1.25x
L15   | 70            |      75.78% |      19.71% |  1.25x
L20   | 70            |      76.81% |      19.71% |  1.25x
L25   | 74            |      76.50% |      19.71% |  1.25x
-----------------------------------------------------------------
All 26 layers compressed and injected successfully.
time: 29.54s
cummulative_time: 46.79s


## Step 3: Full-Model Evaluation on GLUE MNLI (All 26 Layers Compressed)

We evaluate the full model on GLUE MNLI with all 26 MLP `gate_proj` layers concurrently compressed via Branch 1.

In [6]:
# =====================================================================
# STEP 6: Full-Model MNLI Benchmark with All 26 Layers Compressed
# =====================================================================
print("\nEvaluating Full Model (All 26 Layers Compressed - Branch 1) on GLUE MNLI...")
comp_preds, comp_gts = [], []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Evaluating All 26 Layers Compressed"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)
        next_token_logits = outputs.logits[0, -1, :]
        pred_label = torch.argmax(next_token_logits[label_token_ids]).item()
        comp_preds.append(pred_label)
        comp_gts.append(sample["label"])

compressed_all_accuracy = accuracy_score(comp_gts, comp_preds)
global_delta = compressed_all_accuracy - baseline_accuracy

print(f"\nAll-26-Layers Compressed Accuracy: {compressed_all_accuracy * 100:.2f}%")
print(f"Delta vs Uncompressed Baseline:    {global_delta * 100:+.2f}%")

# Restore clean baseline weights for all layers
for l in range(NUM_LAYERS):
    model.model.layers[l].mlp.gate_proj.weight.data = W_gate_orig_all[l].clone()
print("All 26 layer baseline weights restored.")


Evaluating Full Model (All 26 Layers Compressed - Branch 1) on GLUE MNLI...


Evaluating All 26 Layers Compressed: 100%|██████████| 150/150 [00:05<00:00, 27.81it/s]


All-26-Layers Compressed Accuracy: 41.33%
Delta vs Uncompressed Baseline:    -9.33%
All 26 layer baseline weights restored.
time: 5.40s
cummulative_time: 52.19s


## Step 4: Aggregate Parameter Reduction Accounting & Global Time Log

We tabulate overall model parameter reduction across all 26 layers, average reconstruction error, and final benchmark retention.

In [7]:
# =====================================================================
# STEP 7: Aggregate Parameter Reduction & Export Artifact
# =====================================================================
global_orig_params = total_orig_params_gate
global_comp_params = total_comp_params_gate
global_cut_pct = (1.0 - global_comp_params / global_orig_params) * 100.0
global_ratio = global_orig_params / global_comp_params
mean_recon_error = np.mean([r["recon_error"] for r in layer_compression_records])

print("=" * 90)
print(f"{'Metric':<35} | {'Value':<50}")
print("=" * 90)
print(f"{'Decomposition Pipeline':<35} | Branch 1 (Mutually Exclusive Frequency Tucker)")
print(f"{'Total MLP Layers Compressed':<35} | {NUM_LAYERS} layers (gate_proj)")
print(f"{'Tucker Ranks per Layer':<35} | {TUCKER_RANKS}")
print(f"{'Original Gate Params (26 Layers)':<35} | {global_orig_params:,} parameters")
print(f"{'Compressed Gate Params (26 Layers)':<35} | {global_comp_params:,} parameters")
print(f"{'Global Parameters Cut':<35} | {global_orig_params - global_comp_params:,} ({global_cut_pct:.2f}%)")
print(f"{'Global Compression Ratio':<35} | {global_ratio:.2f}x")
print(f"{'Mean Layer Reconstruction Error':<35} | {mean_recon_error * 100:.2f}%")
print(f"{'Baseline Uncompressed Accuracy':<35} | {baseline_accuracy * 100:.2f}%")
print(f"{'All-26-Layers Compressed Accuracy':<35} | {compressed_all_accuracy * 100:.2f}%")
print(f"{'Accuracy Delta':<35} | {global_delta * 100:+.2f}%")
print("=" * 90)

# Global Notebook Timing
total_notebook_runtime = time.time() - GLOBAL_NOTEBOOK_START_TIME
print(f"cummulative_time: {total_notebook_runtime:.2f}s")

# Save export artifact
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
output_path = artifact_dir / "07_all_layers_branch1_results.json"

export_data = {
    "experiment": "07_all_layers_freq_tucker_branch1",
    "model_id": model_id,
    "num_layers_compressed": NUM_LAYERS,
    "ranks": TUCKER_RANKS,
    "eval_samples": len(eval_data),
    "global_orig_params": global_orig_params,
    "global_comp_params": global_comp_params,
    "global_cut_pct": round(global_cut_pct, 2),
    "global_compression_ratio": round(global_ratio, 3),
    "mean_recon_error_pct": round(mean_recon_error * 100, 2),
    "baseline_accuracy": baseline_accuracy,
    "compressed_accuracy": compressed_all_accuracy,
    "accuracy_delta": global_delta,
    "layer_details": layer_compression_records,
    "timing_summary": {
        "cummulative_time_sec": round(total_notebook_runtime, 3),
        "cell_timings": NOTEBOOK_TIMINGS,
    }
}

save_json_metrics(export_data, output_path)
print(f"\nAll-26-layers Branch 1 results saved to: {output_path}")

Metric                              | Value                                             
Decomposition Pipeline              | Branch 1 (Mutually Exclusive Frequency Tucker)
Total MLP Layers Compressed         | 26 layers (gate_proj)
Tucker Ranks per Layer              | [4, 180, 600]
Original Gate Params (26 Layers)    | 207,028,224 parameters
Compressed Gate Params (26 Layers)  | 166,219,248 parameters
Global Parameters Cut               | 40,808,976 (19.71%)
Global Compression Ratio            | 1.25x
Mean Layer Reconstruction Error     | 76.34%
Baseline Uncompressed Accuracy      | 50.67%
All-26-Layers Compressed Accuracy   | 41.33%
Accuracy Delta                      | -9.33%
cummulative_time: 52.20s

All-26-layers Branch 1 results saved to: artifacts/07_all_layers_branch1_results.json
time: 0.00s
cummulative_time: 52.20s
